# 0. Preparación del ambiente – Proyecto Databricks

**Objetivo:** preparar y validar los recursos que utilizará el proceso ETL con arquitectura Medallion.

Este notebook sigue la organización del notebook de preparación visto en clase, adaptando los nombres a los recursos que ya existen en este proyecto.

**Recursos del proyecto**
- Storage Account: `stproyectofinalehs`
- Contenedor de entrada: `raw`
- Contenedor de destino: `datalake`
- Storage Credential: `cred_adls_proyecto_final`
- External Locations: `extloc_adls_raw` y `extloc_adls_datalake`
- Catálogo: `dbw_proyecto_final_ehs`
- Esquemas: `bronze`, `silver` y `golden`

> **Importante:** este notebook es seguro para ejecutar varias veces. No elimina catálogos, esquemas, tablas ni archivos. La limpieza completa se realizará de forma separada en `1_Drop_Medallion.ipynb`.

## 1. Parámetros del ambiente

Se parametriza el nombre del Storage Account, siguiendo el uso de widgets visto en clase. El valor por defecto corresponde al recurso creado para este proyecto.

In [ ]:
dbutils.widgets.text("storageName", "stproyectofinalehs", "Storage Account")

STORAGE_NAME = dbutils.widgets.get("storageName").strip()

if not STORAGE_NAME:
    raise ValueError("El parámetro storageName no puede estar vacío.")

print(f"Storage Account configurado: {STORAGE_NAME}")

## 2. Configuración de catálogo y rutas

Se centralizan los nombres del catálogo, esquemas y rutas ADLS Gen2. En este proyecto Bronze, Silver y Golden se almacenan como carpetas dentro del contenedor `datalake`; no se crean contenedores separados para cada capa.

In [ ]:
CATALOGO = "dbw_proyecto_final_ehs"

ESQUEMAS = ["bronze", "silver", "golden"]

RAW_PATH = f"abfss://raw@{STORAGE_NAME}.dfs.core.windows.net/"
DATALAKE_PATH = f"abfss://datalake@{STORAGE_NAME}.dfs.core.windows.net/"

BRONZE_PATH = f"{DATALAKE_PATH}bronze/"
SILVER_PATH = f"{DATALAKE_PATH}silver/"
GOLDEN_PATH = f"{DATALAKE_PATH}golden/"

STORAGE_CREDENTIAL = "cred_adls_proyecto_final"
EXTERNAL_LOCATION_RAW = "extloc_adls_raw"
EXTERNAL_LOCATION_DATALAKE = "extloc_adls_datalake"

print("Catálogo:", CATALOGO)
print("Raw:", RAW_PATH)
print("Datalake:", DATALAKE_PATH)
print("Bronze:", BRONZE_PATH)
print("Silver:", SILVER_PATH)
print("Golden:", GOLDEN_PATH)

## 3. Creación y validación de Storage Credential y External Locations

En clase se utilizan instrucciones SQL para crear las External Locations a partir de una Storage Credential existente.

En nuestro proyecto, la Storage Credential `cred_adls_proyecto_final` ya fue creada en Unity Catalog utilizando la Managed Identity del Access Connector. Por ello, primero verificamos que esté registrada y luego declaramos las dos External Locations con `CREATE EXTERNAL LOCATION IF NOT EXISTS`.

- `extloc_adls_raw`: acceso al contenedor de origen `raw`.
- `extloc_adls_datalake`: acceso al contenedor de destino `datalake`.

`IF NOT EXISTS` permite ejecutar nuevamente estas instrucciones sin generar un error cuando la ubicación ya existe. Si existe, Databricks no la modifica; por eso, al final consultamos las External Locations para verificar su registro.

> **Nota:** la Storage Credential se creó previamente desde Unity Catalog con Azure Managed Identity. No la recreamos aquí para evitar duplicar credenciales o modificar la identidad asociada.

In [ ]:
# Verificar que la Storage Credential utilizada por el proyecto esté registrada.
display(spark.sql("SHOW STORAGE CREDENTIALS"))

In [ ]:
# Crear la External Location de Raw si todavía no existe.
# Esta ubicación corresponde al contenedor de entrada y utiliza la Storage Credential del proyecto.

sql_extloc_raw = f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {EXTERNAL_LOCATION_RAW}
URL '{RAW_PATH}'
WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
COMMENT 'Ubicacion de origen Raw del proyecto final Databricks'
"""

spark.sql(sql_extloc_raw)

print(f"External Location verificada: {EXTERNAL_LOCATION_RAW}")
print(f"Ruta: {RAW_PATH}")

In [ ]:
# Crear la External Location de Datalake si todavía no existe.
# Esta ubicación se utilizará para almacenar las capas Bronze, Silver y Golden.

sql_extloc_datalake = f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {EXTERNAL_LOCATION_DATALAKE}
URL '{DATALAKE_PATH}'
WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL})
COMMENT 'Ubicacion de destino Datalake para las capas Medallion'
"""

spark.sql(sql_extloc_datalake)

print(f"External Location verificada: {EXTERNAL_LOCATION_DATALAKE}")
print(f"Ruta: {DATALAKE_PATH}")

### Verificación de External Locations

Se consulta Unity Catalog para comprobar que ambas ubicaciones estén registradas. La ubicación `extloc_adls_raw` ya tiene configurado el acceso de solo lectura en nuestro ambiente; la instrucción `IF NOT EXISTS` no altera esa configuración existente.

In [ ]:
# Verificar las External Locations registradas en Unity Catalog.
display(spark.sql("SHOW EXTERNAL LOCATIONS"))

## 4. Validación de acceso a ADLS Gen2

Se comprueba el acceso a los contenedores `raw` y `datalake` utilizando las rutas `abfss://` configuradas para el proyecto.

- **Raw:** se valida la lectura del listado de archivos CSV de origen.
- **Datalake:** se valida el acceso al contenedor donde se almacenan las capas Bronze, Silver y Golden.

Estas operaciones son de lectura; no modifican ni eliminan archivos.

In [ ]:
print("Archivos disponibles en Raw:")
display(dbutils.fs.ls(RAW_PATH))

In [ ]:
print("Contenido actual del contenedor Datalake:")
display(dbutils.fs.ls(DATALAKE_PATH))

## 5. Selección del catálogo de Unity Catalog

Se selecciona el catálogo creado para el proyecto. Esta instrucción define el contexto para las operaciones SQL posteriores.

In [ ]:
spark.sql(f"USE CATALOG `{CATALOGO}`")
print(f"Catálogo activo: {CATALOGO}")

## 6. Creación de esquemas Medallion

Se crean los esquemas Bronze, Silver y Golden si todavía no existen.

Se utiliza `IF NOT EXISTS` para que esta sección pueda ejecutarse repetidamente sin eliminar ni reemplazar los esquemas existentes. La capa Raw permanece como contenedor de origen en ADLS y no se registra como esquema de este catálogo.

In [ ]:
for esquema in ESQUEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOGO}`.`{esquema}`")
    print(f"Esquema verificado: {CATALOGO}.{esquema}")

## 7. Validación de esquemas y ubicaciones físicas

Se consulta Unity Catalog para comprobar los esquemas y se revisan las carpetas de cada capa en Datalake.

Es normal que las carpetas Silver y Golden todavía no contengan archivos Delta: se poblarán cuando se ejecuten sus respectivos notebooks de transformación.

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN `{CATALOGO}`"))

In [ ]:
rutas_capas = {
    "Bronze": BRONZE_PATH,
    "Silver": SILVER_PATH,
    "Golden": GOLDEN_PATH
}

for capa, ruta in rutas_capas.items():
    print(f"\nValidando capa {capa}: {ruta}")

    try:
        contenido = dbutils.fs.ls(ruta)
        print(f"Carpeta {capa} encontrada.")
        display(contenido)

    except Exception as e:
        mensaje_error = str(e)

        if "PathNotFound" in mensaje_error or "The specified path does not exist" in mensaje_error:
            print(f"Carpeta {capa} todavía no creada. Se generará durante su proceso ETL.")
        else:
            raise

## 8. Validación de tablas Bronze

Se comprueba la disponibilidad de las cinco tablas Bronze generadas por el notebook de ingesta. También se comparan sus conteos con los conteos de referencia obtenidos durante la ingesta inicial.

Esta sección no vuelve a ingerir ni sobrescribe datos: únicamente lee las tablas y valida sus registros.

In [ ]:
TABLAS_BRONZE = {
    "almacenes": 20,
    "clientes": 10012,
    "inventario": 10000,
    "productos": 1000,
    "ventas": 100020
}

resultados_bronze = []

for nombre_tabla, conteo_esperado in TABLAS_BRONZE.items():
    tabla_completa = f"{CATALOGO}.bronze.{nombre_tabla}"
    conteo_actual = spark.table(tabla_completa).count()

    resultados_bronze.append(
        (nombre_tabla, conteo_esperado, conteo_actual,
         conteo_actual == conteo_esperado)
    )

df_validacion_bronze = spark.createDataFrame(
    resultados_bronze,
    ["tabla", "conteo_esperado", "conteo_actual", "validacion_ok"]
)

display(df_validacion_bronze)

if any(not fila[3] for fila in resultados_bronze):
    raise AssertionError("Una o más tablas Bronze no coinciden con los conteos de referencia.")

print("Validación Bronze completada.")

## 9. Resumen de preparación

Al finalizar este notebook deben estar disponibles:

- Acceso de lectura al contenedor Raw mediante Managed Identity.
- Acceso al contenedor Datalake mediante la External Location configurada.
- Catálogo `dbw_proyecto_final_ehs`.
- Esquemas `bronze`, `silver` y `golden`.
- Cinco tablas Bronze accesibles y con sus conteos de referencia.

**Siguiente etapa:** ejecutar `2_Ingesta_Bronze.ipynb` cuando se desee reconstruir la ingesta, o continuar con el notebook de transformación Silver cuando Bronze ya esté validado.

**Nota:** la creación de tablas Silver y Golden se realizará en los notebooks de transformación, cuando estén definidas sus columnas y reglas de negocio. Así evitamos crear estructuras vacías o duplicar definiciones.

In [ ]:
# Resumen de las validaciones realizadas en este notebook.

total_registros_bronze = sum(fila[2] for fila in resultados_bronze)
bronze_validado = all(fila[3] for fila in resultados_bronze)

print("RESUMEN DE PREPARACIÓN DEL AMBIENTE")
print("-----------------------------------")
print(f"Catálogo: {CATALOGO}")
print(f"Esquemas configurados: {', '.join(ESQUEMAS)}")
print(f"Storage Credential: {STORAGE_CREDENTIAL}")
print(f"External Locations: {EXTERNAL_LOCATION_RAW}, {EXTERNAL_LOCATION_DATALAKE}")
print(f"Tablas Bronze validadas: {len(resultados_bronze)} de {len(TABLAS_BRONZE)}")
print(f"Registros Bronze verificados: {total_registros_bronze:,}")
print(f"Validación de conteos Bronze: {'CORRECTA' if bronze_validado else 'CON DIFERENCIAS'}")

if not bronze_validado:
    raise AssertionError("La validación final de Bronze no fue satisfactoria.")

print("\nPreparación del ambiente completada.")